# Court E2E — BGE-M3 hybrid + case-fixed statute pivot (T4-compatible)

**Drops from v1/v2**: Qwen3-Embedding-8B (failed to surface gold), Qwen3-Reranker-8B (flat scores on Swiss court paragraphs), BGE echo channel (8/10 queries had 0 hits), noun-phrase BM25 (marginal).

**Keeps + adds**:
1. **BGE-M3 hybrid** (568M params, T4-compatible). NOWJ@COLIEE 2025 used this to win Legal Case Entailment. Dense + sparse + ColBERT all from one model.
2. **Case-fixed statute pivot** using law pipeline output (lowercase code normalization fixes the StPO/STPO mismatch).
3. **Qwen3-14B judge** with rich context on top 50 borderline (auto-YES top 5, judge the rest).

**Realistic F1 target on val court: 0.25-0.40.** Honest about it. The reranker bottleneck is the binding constraint.

## Cell 1 — Install + setup

In [ ]:
!pip install -q FlagEmbedding peft accelerate transformers sentencepiece pyarrow vllm autoawq

import torch, time, json, gc, os, re, sys, io
import pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

from google.colab import drive
drive.mount('/content/drive')

SWISS_LAW_DIR = Path('/content/drive/MyDrive/swiss_law')
LAW_PIPE_DIR  = Path('/content/drive/MyDrive/Omnilex-Agentic-Retrieval-Competition')
DATA_DIR  = SWISS_LAW_DIR / 'data'
OUT_DIR   = SWISS_LAW_DIR / 'court_e2e_bgem3_2026-05-22'
OUT_DIR.mkdir(parents=True, exist_ok=True)
BGE_INDEX_DIR = OUT_DIR / 'bge_m3_index'
BGE_INDEX_DIR.mkdir(exist_ok=True)

LAW_PREDS_JSON = LAW_PIPE_DIR / 'retrieval' / 'pipeline_output_v12' / 'law_predictions_per_query.json'

print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Cell 2 — Load val + law preds + court corpus

In [ ]:
val = pd.read_csv(DATA_DIR / 'val.csv')
QID, QCOL, GOLDCOL = 'query_id', 'query', 'gold_citations'
val['gold_list'] = val[GOLDCOL].apply(lambda s: [c.strip() for c in re.split(r'[;,]', str(s)) if c.strip()])

law_preds = json.loads(LAW_PREDS_JSON.read_text()) if LAW_PREDS_JSON.exists() else {}
print(f'Loaded law predictions: {len(law_preds)} queries, avg {np.mean([len(v) for v in law_preds.values()]):.1f} statutes/query')

court = pd.read_csv(DATA_DIR / 'court_considerations.csv', low_memory=False)
court['text'] = court['text'].astype(str)
court_cit_set = set(court['citation'])
val['gold_court'] = val['gold_list'].apply(lambda L: [c for c in L if c in court_cit_set])
print('Court corpus rows:', len(court), '| mean court gold/query:', val['gold_court'].apply(len).mean())

## Cell 3 — Filter to doctrinal pool

In [ ]:
CANTONAL_RE   = re.compile(r'^(KGer|OGer|BezGer|Kantonsgericht|Cour cantonale|Tribunal cantonal)', re.I)
DISPOSITIF_RE = re.compile(
    r'^\s*(?:\d+\.?\s+)?'
    r'(?:Die Beschwerde wird|Le recours est|Il ricorso \u00e8|Im Namen|Au nom de|'
    r'Gerichtskosten\s|Les frais judiciaires|Es werden keine Kosten|'
    r'Il n\'est pas per\u00e7u|Lausanne,)', re.I)

mask = (
    (~court['citation'].str.match(CANTONAL_RE, na=False))
    & (~court['text'].str.match(DISPOSITIF_RE, na=False))
    & (court['text'].str.len() >= 200)
)
pool = court[mask].reset_index(drop=True)
print(f'Doctrinal pool: {len(pool)} / {len(court)} ({100*len(pool)/len(court):.1f}%)')

all_court_gold = set()
for L in val['gold_court']: all_court_gold.update(L)
surv = all_court_gold & set(pool['citation'])
print(f'Court gold surviving filter: {len(surv)}/{len(all_court_gold)} ({100*len(surv)/max(1,len(all_court_gold)):.1f}%)')

## Cell 4 — Build BGE-M3 dense index over the pool

One-time. ~1-2 hours on Blackwell. Saves to Drive for reuse.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

DENSE_PATH    = BGE_INDEX_DIR / 'pool_dense.npy'
POOL_CIT_PATH = BGE_INDEX_DIR / 'pool_citations.json'

if DENSE_PATH.exists() and POOL_CIT_PATH.exists():
    pool_dense = np.load(DENSE_PATH)
    saved_cits = json.loads(POOL_CIT_PATH.read_text())
    print(f'Loaded BGE-M3 dense index: shape={pool_dense.shape}, saved_n={len(saved_cits)}')
    if saved_cits != pool['citation'].tolist():
        print('WARN: saved citations differ from current pool. Re-encoding required.')
        DENSE_PATH.unlink()

if not DENSE_PATH.exists():
    print('Encoding pool with BGE-M3 (dense only, fp16)...')
    model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices='cuda')
    BATCH = 256
    pool_dense = np.zeros((len(pool), 1024), dtype=np.float16)
    t0 = time.time()
    for i in range(0, len(pool), BATCH):
        batch_texts = pool['text'].iloc[i:i+BATCH].tolist()
        out = model.encode(batch_texts, batch_size=64, max_length=512,
                           return_dense=True, return_sparse=False, return_colbert_vecs=False)
        pool_dense[i:i+BATCH] = out['dense_vecs'].astype(np.float16)
        if (i // BATCH) % 50 == 0:
            elapsed = time.time() - t0
            done = i + BATCH
            eta = elapsed / max(done, 1) * (len(pool) - done) / 60
            print(f'  {done}/{len(pool)} ({100*done/len(pool):.1f}%) | elapsed {elapsed/60:.1f}min | ETA {eta:.1f}min')
    np.save(DENSE_PATH, pool_dense)
    POOL_CIT_PATH.write_text(json.dumps(pool['citation'].tolist()))
    print(f'Saved dense index: {DENSE_PATH.name}, total time = {(time.time()-t0)/60:.1f}min')
    del model; gc.collect(); torch.cuda.empty_cache()

# Load to GPU for fast cosine
pool_dense_gpu = torch.from_numpy(pool_dense).to('cuda', dtype=torch.float16)
norms = pool_dense_gpu.norm(dim=1, keepdim=True).clamp_min(1e-6)
pool_dense_gpu = pool_dense_gpu / norms
print(f'pool_dense on GPU: {tuple(pool_dense_gpu.shape)}, VRAM={torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 5 — Channel A: case-fixed statute pivot from law pipeline output

Fix vs v2: **lowercase all code names before indexing and lookup.** This fixes the `STPO` vs `StPO` mismatch.

In [ ]:
STATUTE_RE = re.compile(
    r'\b[Aa]rt(?:icle|icolo|\.)?\.?\s+(\d+[a-z]?)'
    r'(?:\s+(?:Abs|al|cpv|para|paragraph)\.?\s+(\d+))?'
    r'(?:\s+(?:lit|let|lett)\.?\s+([a-z]))?'
    r'\s+([A-Z][A-Za-z]{1,7}\d?)\b'
)

STATUTE_ALIASES_LOWER = {
    'stpo':['cpp'],   'cpp':['stpo'],
    'stgb':['cp'],    'cp':['stgb'],
    'zgb':['cc'],     'cc':['zgb'],
    'or':['co'],      'co':['or'],
    'zpo':['cpc'],    'cpc':['zpo'],
    'bgg':['ltf'],    'ltf':['bgg'],
    'bv':['cst','cost'], 'cst':['bv','cost'], 'cost':['bv','cst'],
    'atsg':['lpga'],  'lpga':['atsg'],
    'ivg':['lai'],    'lai':['ivg'],
    'uvg':['laa'],    'laa':['uvg'],
    'schkg':['lp'],   'lp':['schkg'],
    'emrk':['cedh'],  'cedh':['emrk'],
}

def parse_st(s):
    m = STATUTE_RE.search(s)
    if not m: return None
    art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4).lower()
    full = f'art. {art}'
    if abs_: full += f' abs. {abs_}'
    if lit:  full += f' lit. {lit}'
    full += f' {code}'
    return full, f'art. {art} {code}', code

def aliases(art_code_lower):
    m = re.match(r'(art\.\s+\d+[a-z]?)\s+(\S+)', art_code_lower)
    if not m: return [art_code_lower]
    art, code = m.group(1), m.group(2)
    out = [art_code_lower]
    for alias in STATUTE_ALIASES_LOWER.get(code, []):
        out.append(f'{art} {alias}')
    return out

# Build pool inverted index (lowercase everything)
print('Indexing pool statutes (case-normalized)...')
ac_idx, full_idx = defaultdict(list), defaultdict(list)
t0 = time.time()
for i in range(len(pool)):
    text = pool['text'].iloc[i][:3000]
    for m in STATUTE_RE.finditer(text):
        art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4).lower()
        ac = f'art. {art} {code}'
        ac_idx[ac].append(i)
        full = f'art. {art}'
        if abs_: full += f' abs. {abs_}'
        if lit:  full += f' lit. {lit}'
        full += f' {code}'
        full_idx[full].append(i)
print(f'  Built in {time.time()-t0:.1f}s. {len(ac_idx)} art+code keys, {len(full_idx)} full-spec keys.')

ch_a = {}
for _, r in val.iterrows():
    qid = r[QID]
    # Query statutes = union of law preds + statutes in query text itself
    raw_statutes = list(law_preds.get(qid, [])) + [m.group(0) for m in STATUTE_RE.finditer(r[QCOL])]
    full_set, ac_set = set(), set()
    for s in raw_statutes:
        p = parse_st(s)
        if p:
            full_set.add(p[0])
            for ac in aliases(p[1]):
                ac_set.add(ac)
    doc_score = defaultdict(int)
    for s in full_set:
        for i in full_idx.get(s, []):
            doc_score[i] += 3
    for ac in ac_set:
        for i in ac_idx.get(ac, []):
            doc_score[i] += 1
    top = sorted(doc_score.items(), key=lambda x: -x[1])[:1000]
    ch_a[qid] = {'idx': [i for i, _ in top], 'score': [s for _, s in top]}
    gold = set(r['gold_court'])
    hit = sum(pool['citation'].iloc[i] in gold for i, _ in top)
    print(f"  {qid}: {len(full_set)} full + {len(ac_set)} art+code -> {len(top)} cands | gold_hit = {hit}/{len(gold)}")

## Cell 6 — Channel B: BGE-M3 dense top-3000 per query

In [ ]:
# Load BGE-M3 for query encoding (small, fast)
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices='cuda')

ch_b = {}
for _, r in val.iterrows():
    qid = r[QID]
    q_out = model.encode([r[QCOL]], batch_size=1, max_length=8192,
                         return_dense=True, return_sparse=False, return_colbert_vecs=False)
    q_vec = torch.from_numpy(q_out['dense_vecs']).to('cuda', dtype=torch.float16)
    q_vec = q_vec / q_vec.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    sims = (pool_dense_gpu @ q_vec.T).squeeze()
    top_k = torch.topk(sims, k=3000).indices.cpu().numpy()
    top_s = sims[top_k].cpu().numpy()
    ch_b[qid] = {'idx': top_k.tolist(), 'score': top_s.tolist()}
    gold = set(r['gold_court'])
    hit = sum(pool['citation'].iloc[i] in gold for i in top_k)
    print(f"  {qid}: BGE-M3 dense top-3000 | gold_hit = {hit}/{len(gold)}")

del model; gc.collect(); torch.cuda.empty_cache()

## Cell 7 — Union + Stage 1 recall gate

In [ ]:
union_idx = {}
stage1_recall = []
for _, r in val.iterrows():
    qid = r[QID]
    u = set(ch_a.get(qid, {}).get('idx', [])) | set(ch_b.get(qid, {}).get('idx', []))
    union_idx[qid] = list(u)
    gold = set(r['gold_court'])
    if gold:
        hit = sum(pool['citation'].iloc[i] in gold for i in u)
        stage1_recall.append(hit / len(gold))
        print(f"  {qid}: union={len(u)}, gold_in_pool={hit}/{len(gold)} (R={hit/len(gold):.2f})")
print(f'\n=== Stage 1 macro recall: {np.mean(stage1_recall):.3f} ===')
print('   target: >= 0.55 to make F1 0.30+ reachable; >= 0.70 to make 0.40+ reachable')

## Cell 8 — BGE-M3 HYBRID rerank on union (dense + sparse + ColBERT)

Per query: compute hybrid score for each union candidate using all three modes. Weights 0.4/0.2/0.4 are BGE-M3 paper recommendation.

In [ ]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices='cuda')

hybrid_results = {}
for _, r in val.iterrows():
    qid = r[QID]
    cand_idxs = union_idx[qid]
    pairs = [[r[QCOL], pool['text'].iloc[i][:3000]] for i in cand_idxs]
    t0 = time.time()
    # compute_score returns dict with 'colbert+sparse+dense' and individual scores
    scores_out = model.compute_score(pairs, max_passage_length=512,
                                      weights_for_different_modes=[0.4, 0.2, 0.4])
    hyb = scores_out['colbert+sparse+dense']
    df = pd.DataFrame({
        'pool_idx': cand_idxs,
        'citation': [pool['citation'].iloc[i] for i in cand_idxs],
        'text':     [pool['text'].iloc[i] for i in cand_idxs],
        'hybrid':   hyb,
        'dense':    scores_out['dense'],
        'sparse':   scores_out['sparse'],
        'colbert':  scores_out['colbert'],
    }).sort_values('hybrid', ascending=False).reset_index(drop=True)
    hybrid_results[qid] = df
    gold = set(r['gold_court'])
    top50_hit = sum(c in gold for c in df.head(50)['citation'])
    print(f"  {qid}: hybrid-reranked {len(df)} in {time.time()-t0:.1f}s | gold in top50 = {top50_hit}/{len(gold)}")

del model; gc.collect(); torch.cuda.empty_cache()

## Cell 9 — Hybrid-only F1 sweep (checkpoint)

In [ ]:
def f1(p, g):
    if not g: return None
    tp = len(p & g)
    if not p or tp == 0: return 0.0
    P = tp/len(p); R = tp/len(g)
    return 2*P*R/(P+R)

print(f"{'K':>4} | {'macro F1':>10} | per-query")
best = (0, 0.0)
for K in [5, 8, 10, 12, 15, 20, 25, 30, 50, 75]:
    per_q = []
    for _, r in val.iterrows():
        pred = set(hybrid_results[r[QID]].head(K)['citation'])
        v = f1(pred, set(r['gold_court']))
        if v is not None: per_q.append(v)
    m = np.mean(per_q)
    if m > best[1]: best = (K, m)
    print(f'{K:>4} | {m:>10.3f} | {[round(v,2) for v in per_q]}')
print(f'\nHybrid-only best fixed-K: K={best[0]}, macro F1 = {best[1]:.3f}')

## Cell 10 — Qwen3-14B judge on borderline (top 50 → top 5 auto-YES + 45 judged)

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

QWEN_NAME = 'Qwen/Qwen3-14B-AWQ'
llm = LLM(model=QWEN_NAME, quantization='awq_marlin', max_model_len=4096,
          gpu_memory_utilization=0.85, dtype='float16', trust_remote_code=True)
jtok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)

DE_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?(Der Beschwerdef\u00fchrer|Die Beschwerdef\u00fchrerin|Die Vorinstanz)', re.I)
FR_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?(Le recourant|La recourante|La cour cantonale)', re.I)
CHAMBER_RE  = re.compile(r'\b(BGE|ATF|DTF)\s+\d+\s+([IVX]+)|^(\d[A-Z])_')

def feats(text, citation):
    statutes = sorted({m.group(0) for m in STATUTE_RE.finditer(text[:1500])})[:6]
    voice = 'court'
    if DE_PARTY_RE.match(text) or FR_PARTY_RE.match(text):
        voice = 'appellant_or_lower_court'
    m = CHAMBER_RE.search(citation)
    chamber = (m.group(2) or m.group(3)) if m else ''
    return statutes, voice, chamber

def judge_prompt(query, citation, text, fts, law_preds_list):
    statutes, voice, chamber = fts
    overlap = sorted(set(statutes) & set(law_preds_list))
    msgs = [
        {'role':'system', 'content':'You are a Swiss Federal Court doctrinal-paragraph expert. Output exactly one token: YES or NO. Default to NO when uncertain.'},
        {'role':'user', 'content':
            f'Query (English):\n{query[:1500]}\n\n'
            f'Predicted law citations for this query: {law_preds_list[:10]}\n\n'
            f'Candidate court paragraph:\n'
            f'  Citation: {citation}\n'
            f'  Chamber: {chamber}\n'
            f'  Opener voice: {voice}\n'
            f'  Statutes cited in this paragraph: {statutes}\n'
            f'  Statutes overlapping with query: {overlap}\n'
            f'  Text: {text[:1800]}\n\n'
            'Answer YES only if ALL three:\n'
            '(a) Federal court (not cantonal)\n'
            '(b) States/echoes/applies a doctrinal rule relevant to the query (rule, echo, application, party_position, or lower_court_summary all acceptable; pure dispositif/cost/signature are not)\n'
            '(c) The doctrinal sub-question matches the query (not merely the same statute used in unrelated context)\n\n'
            'Answer:'
        }
    ]
    return jtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

TOP_KEEP = 50
AUTO_YES_K = 5
auto_yes = {}; borderline = {}
for _, r in val.iterrows():
    qid = r[QID]
    top = hybrid_results[qid].head(TOP_KEEP).reset_index(drop=True)
    auto_yes[qid] = set(top.head(AUTO_YES_K)['citation'])
    borderline[qid] = top.iloc[AUTO_YES_K:].reset_index(drop=True)

all_prompts, all_keys = [], []
for qid, bdf in borderline.items():
    q = val[val[QID]==qid][QCOL].iloc[0]
    lp = law_preds.get(qid, [])
    for _, c in bdf.iterrows():
        all_prompts.append(judge_prompt(q, c['citation'], c['text'], feats(c['text'], c['citation']), lp))
        all_keys.append((qid, c['citation']))

print(f'Judging {len(all_prompts)} borderline candidates...')
t0 = time.time()
outs = llm.generate(all_prompts, SamplingParams(temperature=0.0, max_tokens=4))
print(f'Done in {time.time()-t0:.1f}s')

verdicts = {}
for (qid, cit), o in zip(all_keys, outs):
    t = o.outputs[0].text.strip().upper()
    verdicts.setdefault(qid, {})[cit] = t.startswith('YES')

del llm; gc.collect(); torch.cuda.empty_cache()

## Cell 11 — Final F1 + error analysis

In [ ]:
rows = []
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold:
        rows.append({'qid':qid,'gold':0,'pred':0,'tp':0,'P':None,'R':None,'F1':None}); continue
    pred = set(auto_yes[qid]) | {c for c,v in verdicts.get(qid,{}).items() if v}
    if len(pred) < 3:
        for c in hybrid_results[qid]['citation']:
            pred.add(c)
            if len(pred) >= 3: break
    tp = len(pred & gold); P = tp/max(1,len(pred)); R = tp/len(gold); F = 2*P*R/(P+R) if (P+R) else 0.0
    rows.append({'qid':qid,'gold':len(gold),'pred':len(pred),'tp':tp,'P':round(P,3),'R':round(R,3),'F1':round(F,3)})

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / 'per_query_results.csv', index=False)
macro = res.loc[res['F1'].notna(), ['P','R','F1']].mean()
print(res.to_string(index=False))
print(f"\n=== MACRO: P={macro['P']:.3f}  R={macro['R']:.3f}  F1={macro['F1']:.3f} ===")

print('\n--- Gold ranks in hybrid pool per query ---')
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold: continue
    df = hybrid_results[qid].reset_index(drop=True)
    in_pool = df[df['citation'].isin(gold)]
    ranks = (in_pool.index + 1).tolist()
    miss = gold - set(df['citation'])
    print(f'{qid}: {len(in_pool)}/{len(gold)} | ranks={ranks[:20]} | missed={len(miss)}')